# 北國新聞の記事検索結果を取得する

北國新聞 DIGITALで「能登半島地震」を検索し、最終ページまでタイトル・本文・公開日時・URLを取得します。

- 検索画面は「もっと見る」ではなく番号付きページングで、合計件数は表示されません。`次ページ` がなくなるまで巡回して全件を取得します。
- 北國新聞 DIGITALへログインし、契約上閲覧できる本文だけを取得します。アクセス制限の回避は行いません。
- 短時間に大量アクセスしないよう、検索ページと記事ごとに待機時間を設けています。
- 実行前に、利用規約・著作権・robots.txtと契約内容を確認してください。
- サイトの画面構成が変わった場合は、セレクタの調整が必要になることがあります。

In [ ]:
# 初回だけ実行してください
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "requests", "beautifulsoup4", "pandas", "playwright"
])

In [ ]:
import json
import time
from pathlib import Path
from urllib.parse import quote, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.hokkoku.co.jp"
SEARCH_URL = f"{BASE_URL}/list/search"
KEYWORD = "能登半島地震"
SEARCH_PAGE_INTERVAL = 1.0
ARTICLE_INTERVAL = 1.5
TIMEOUT = 30
MAX_SEARCH_PAGES = 1000  # 異常時の無限ループ防止

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
})

## ログイン情報の入力

IDとパスワードは実行中の変数にだけ保持され、ノートブックファイルには保存されません。パスワードは入力中も画面に表示されません。

In [ ]:
from getpass import getpass

HOKKOKU_ID = input("北國新聞 DIGITALのログインID（メールアドレス）: ").strip()
HOKKOKU_PASSWORD = getpass("パスワード: ")

In [ ]:
from playwright.async_api import async_playwright


async def login_to_hokkoku(requests_session, login_id, password):
    """Chromeでログインし、北國新聞ドメインのCookieをrequestsへ渡す。"""
    return_to = quote(f"{SEARCH_URL}?fulltext={quote(KEYWORD)}&scode=hkk&gsign=yes", safe="")
    login_url = f"{BASE_URL}/auth/login?return_to={return_to}"

    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(channel="chrome", headless=False)
    context = await browser.new_context(locale="ja-JP")
    page = await context.new_page()

    try:
        await page.goto(login_url, wait_until="domcontentloaded", timeout=60000)
        email = page.locator('input[name="uidemail"]')
        password_input = page.locator('input[name="password"]')
        submit = page.locator('input[type="submit"][value="ログイン"]')

        if await email.count() != 1 or await password_input.count() != 1:
            raise RuntimeError("北國新聞のログイン入力欄を一意に特定できませんでした。")
        await email.fill(login_id)
        await password_input.fill(password)
        if await submit.count() != 1:
            raise RuntimeError("北國新聞のログインボタンを一意に特定できませんでした。")
        await submit.click()
        await page.wait_for_timeout(1500)

        # 追加認証、アカウント確認、エラー表示がある場合に手動で対応できるようにする。
        if "auth.hokkoku.co.jp" in page.url or "/auth/login" in page.url:
            input(
                "Chromeでログイン（必要なら追加認証）を完了してから、"
                "ここで Enter を押してください: "
            )

        await page.goto(BASE_URL, wait_until="domcontentloaded", timeout=60000)
        page_text = await page.locator("body").inner_text()
        if "ログアウト" not in page_text and "マイページ" not in page_text:
            print("注意: ログイン表示を確認できませんでした。会員記事で本文取得可否を判定します。")

        for cookie in await context.cookies():
            if "hokkoku.co.jp" in cookie.get("domain", ""):
                requests_session.cookies.set(
                    cookie["name"], cookie["value"],
                    domain=cookie.get("domain"), path=cookie.get("path", "/"),
                )
    finally:
        await context.close()
        await browser.close()
        await playwright.stop()


await login_to_hokkoku(session, HOKKOKU_ID, HOKKOKU_PASSWORD)
print("ログインCookieを取得しました。")

In [ ]:
def get_soup(url, *, params=None):
    response = session.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")


def collect_search_urls(keyword):
    """次ページがなくなるまで検索結果を巡回し、記事URLを全件収集する。"""
    urls = []
    seen = set()
    seen_page_urls = set()
    page_url = SEARCH_URL
    params = {"fulltext": keyword, "scode": "hkk"}

    for page_number in range(1, MAX_SEARCH_PAGES + 1):
        soup = get_soup(page_url, params=params)
        canonical_page = page_url if params is None else f"{page_url}?page={page_number}"
        if canonical_page in seen_page_urls:
            raise RuntimeError(f"検索ページが循環しました: {canonical_page}")
        seen_page_urls.add(canonical_page)

        page_items = soup.select('.m-articles > a[href^="/articles/-/"]')
        if not page_items:
            if page_number == 1:
                heading = soup.select_one("main h2")
                heading_text = heading.get_text(" ", strip=True) if heading else ""
                if keyword not in heading_text:
                    raise RuntimeError("検索結果ページを確認できませんでした。")
            break

        added = 0
        for item in page_items:
            url = urljoin(BASE_URL, item.get("href", ""))
            if url and url not in seen:
                seen.add(url)
                urls.append(url)
                added += 1
        print(f"検索 {page_number}ページ目: +{added}件 / 累計 {len(urls):,}件")

        next_link = soup.select_one(
            'link[rel="next"][href], .c-pagination__item.--next a[href]'
        )
        if next_link is None:
            break
        next_url = urljoin(BASE_URL, next_link.get("href", ""))
        if not next_url or next_url == page_url:
            raise RuntimeError("次ページURLを正しく取得できませんでした。")
        page_url = next_url
        params = None
        time.sleep(SEARCH_PAGE_INTERVAL)
    else:
        raise RuntimeError(
            f"検索ページが上限 {MAX_SEARCH_PAGES} に達しました。処理を中断します。"
        )

    return urls


def _news_article_json_ld(soup):
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        items = data if isinstance(data, list) else [data]
        for item in items:
            if isinstance(item, dict) and item.get("@type") in {"NewsArticle", "Article"}:
                return item
    return {}


def fetch_article(url):
    soup = get_soup(url)
    metadata = _news_article_json_ld(soup)

    title = str(metadata.get("headline") or "").strip()
    if not title:
        heading = soup.select_one("main h1, article h1, h1")
        title = heading.get_text(" ", strip=True) if heading else ""

    published_at = str(metadata.get("datePublished") or "").strip()
    if not published_at:
        published = soup.select_one('time[datetime], meta[property="article:published_time"]')
        if published:
            published_at = published.get("datetime") or published.get("content") or ""

    article_body = soup.select_one(".article-body")
    body_parts = []
    if article_body:
        for node in article_body.find_all(["p", "h2", "h3"], recursive=False):
            text = node.get_text(" ", strip=True)
            if text and text not in body_parts:
                body_parts.append(text)
    body = "\n".join(body_parts)

    login_signage = article_body.select_one(".login-signage") if article_body else None
    body_is_excerpt = login_signage is not None
    if body_is_excerpt:
        extraction_error = "契約またはログイン状態により本文が表示されていません"
    elif not body:
        extraction_error = "本文を取得できませんでした"
    else:
        extraction_error = ""

    return {
        "title": title,
        "body": body,
        "published_at": published_at,
        "url": url,
        "body_is_excerpt": body_is_excerpt,
        "extraction_error": extraction_error,
    }


def collect_articles(urls):
    records = []
    for index, url in enumerate(urls, start=1):
        try:
            record = fetch_article(url)
        except Exception as exc:
            record = {
                "title": "", "body": "", "published_at": "", "url": url,
                "body_is_excerpt": False,
                "extraction_error": f"{type(exc).__name__}: {exc}",
            }
        records.append(record)
        print(f"本文 {index:,}/{len(urls):,}: {record['title'] or url}")
        if index < len(urls):
            time.sleep(ARTICLE_INTERVAL)
    return records

In [ ]:
search_urls = collect_search_urls(KEYWORD)
print(f"検索結果の全ページを走査しました: {len(search_urls):,}件")

articles = collect_articles(search_urls)
df = pd.DataFrame(articles)
if not df.empty:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at", ascending=False, na_position="last")
df = df.reset_index(drop=True)
df.insert(0, "ID", range(1, len(df) + 1))

assert len(df) == len(search_urls), (
    f"検索URL {len(search_urls):,}件に対し、データは {len(df):,}件です。"
)
df

In [ ]:
output_path = Path("hokkoku_能登半島地震.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.resolve()}")